In [5]:

!pip install gensim
import numpy as np
import pandas as pd
import nltk
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans



In [6]:
# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# DATASET
sentences = [
    "Students attend classes and complete assignments regularly",
    "Online learning platforms help students gain knowledge",
    "Teachers guide students in their academic journey",
    "Exams are important for evaluating student performance",
    "Students participate in sports and extracurricular activities",
    "Government announces new education policies",
    "University introduces new admission rules",
    "Education system is improving with digital tools",
    "Students prepare for competitive exams and careers",
    "Colleges organize seminars and workshops for students"
]

# PREPROCESSING
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words]
    return " ".join(filtered)

processed_sentences = [preprocess(s) for s in sentences]

print("Processed Sentences:\n", processed_sentences)

Processed Sentences:
 ['students attend classes complete assignments regularly', 'online learning platforms help students gain knowledge', 'teachers guide students academic journey', 'exams important evaluating student performance', 'students participate sports extracurricular activities', 'government announces new education policies', 'university introduces new admission rules', 'education system improving digital tools', 'students prepare competitive exams careers', 'colleges organize seminars workshops students']


In [8]:
# FEATURE ENGINEERING (TF-IDF)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(processed_sentences).toarray()

print("\nTF-IDF Feature Names:\n", vectorizer.get_feature_names_out())


TF-IDF Feature Names:
 ['academic' 'activities' 'admission' 'announces' 'assignments' 'attend'
 'careers' 'classes' 'colleges' 'competitive' 'complete' 'digital'
 'education' 'evaluating' 'exams' 'extracurricular' 'gain' 'government'
 'guide' 'help' 'important' 'improving' 'introduces' 'journey' 'knowledge'
 'learning' 'new' 'online' 'organize' 'participate' 'performance'
 'platforms' 'policies' 'prepare' 'regularly' 'rules' 'seminars' 'sports'
 'student' 'students' 'system' 'teachers' 'tools' 'university' 'workshops']


In [9]:
# COSINE SIMILARITY (3 SENTENCES)
print("\nCosine Similarity:\n")
s1 = X[0]
s2 = X[1]
s3 = X[2]
print("Sentence 1:", sentences[0])
print("Sentence 2:", sentences[1])
print("Sentence 3:", sentences[2])

print("\nSimilarity (S1 & S2):", cosine_similarity([s1], [s2])[0][0])
print("Similarity (S1 & S3):", cosine_similarity([s1], [s3])[0][0])
print("Similarity (S2 & S3):", cosine_similarity([s2], [s3])[0][0])


Cosine Similarity:

Sentence 1: Students attend classes and complete assignments regularly
Sentence 2: Online learning platforms help students gain knowledge
Sentence 3: Teachers guide students in their academic journey

Similarity (S1 & S2): 0.04997522368165085
Similarity (S1 & S3): 0.060517454511888905
Similarity (S2 & S3): 0.05549723093375731


In [10]:
# TEXT CLASSIFICATION (SUPERVISED - NAIVE BAYES)

from sklearn.naive_bayes import MultinomialNB

print("\nText Classification (Naive Bayes):\n")

# Labels (0 = Student Activities, 1 = Education System)
y = [0, 0, 0, 0, 0, 1, 1, 1, 0, 1]

# Train model
model = MultinomialNB()
model.fit(X, y)

# Test sentence
test_sentence = "Students learn new skills through online education"
test_processed = preprocess(test_sentence)
test_vec = vectorizer.transform([test_processed]).toarray()

prediction = model.predict(test_vec)[0]

label_map = {0: "Student Activities", 1: "Education System"}

print("Test Sentence:", test_sentence)
print("Predicted Category:", label_map[prediction])


Text Classification (Naive Bayes):

Test Sentence: Students learn new skills through online education
Predicted Category: Student Activities


In [11]:
# TEXT CLUSTERING (UNSUPERVISED)

print("\nText Clustering (K-Means):\n")

kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(X)

clusters = kmeans.labels_

cluster_groups = {0: [], 1: []}

for i, label in enumerate(clusters):
    cluster_groups[label].append(sentences[i])

for cluster_id, items in cluster_groups.items():
    print(f"Cluster {cluster_id}:")
    for item in items:
        print("-", item)
    print()


Text Clustering (K-Means):

Cluster 0:
- Exams are important for evaluating student performance
- Government announces new education policies
- University introduces new admission rules
- Education system is improving with digital tools
- Students prepare for competitive exams and careers

Cluster 1:
- Students attend classes and complete assignments regularly
- Online learning platforms help students gain knowledge
- Teachers guide students in their academic journey
- Students participate in sports and extracurricular activities
- Colleges organize seminars and workshops for students

